In [8]:
import random
import string
import numpy as np
from milvus import default_server

from pymilvus import (
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection, AnnSearchRequest, RRFRanker, connections,
)

from pymilvus.model.hybrid import BGEM3EmbeddingFunction
from sentence_transformers import SentenceTransformer

import pandas as pd
import sys
import os
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Add the EDP directory to the Python path
# sys.path.append(os.path.abspath(os.path.join('..', 'EDP')))

In [9]:
from pymilvus import connections

connections.connect("default", host="localhost", port="19530")

if connections.has_connection("default"):
    print("Successfully connected to Milvus")
else:
    print("Failed to connect to Milvus")


Successfully connected to Milvus


In [10]:
# Define the data schema for the new Collection
fields = [
    # Use provided id as primary key
    FieldSchema(name="pk", dtype=DataType.VARCHAR, is_primary=True, max_length=100),
    # Store the original text
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
    # Store dense vectors
    FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=512),  # Ensure the dimension matches your embeddings
]

schema = CollectionSchema(fields,  enable_dynamic_field=False)
col_name = 'sbert_experiment2'
col = Collection(col_name, schema, consistency_level="Strong")

In [11]:
dense_index = {"index_type": "FLAT", "metric_type": "IP"}
col.create_index("dense_vector", dense_index)
col.load()

In [12]:
def merge_text_fields(data):
    for item in data:
        # Merge the fields into 'text', separating them by " | "
        merged_text = " | ".join(filter(None, [item.get('text', ''), 
                                            item.get('provided_data', ''), 
                                            item.get('enriched_data', ''), 
                                            item.get('translated_data', '')]))
        
        # Assign the merged text back to the 'text' field
        item['text'] = merged_text
        
        # Remove the individual fields as they're now part of 'text'
        item.pop('provided_data', None)
        item.pop('enriched_data', None)
        item.pop('translated_data', None)
    
    return data

def sbert_embeddings(batch_data):
    data = merge_text_fields(batch_data)
    print(data[0])
    model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')
    embeddings = model.encode([x['text'] for x in data])

    # Prepare data for insertion
    entities = [
        [item['id'] for item in batch_data], #IDs
        [item['text'] for item in batch_data], #Text
        embeddings #Embeddings
    ]

    return entities

In [13]:
# import os
# import gzip
# import json
# from concurrent.futures import ThreadPoolExecutor, as_completed
# import tqdm 
# import sys

# # Function to load data from a compressed JSON file
# def load_compressed_json(file_path):
#     """Function to load data from a compressed JSON file."""
#     with gzip.open(file_path, 'rt', encoding='utf-8') as f:
#         return json.load(f)

# # Function to load data in 64MB batches for Milvus indexing
# def load_data_in_batches_for_indexing(parsed_directory, max_batch_size_mb=64, max_workers=4):
#     """Load data from compressed JSON files in batches for indexing into Milvus."""
#     max_batch_size_bytes = max_batch_size_mb * 1024 * 1024  # Convert MB to bytes
#     batch_data = []  # To store the current batch
#     batch_size = 0  # To track the size of the current batch in bytes

#     # Collect all the .json.gz file paths
#     file_paths = []
#     for root, dirs, files in os.walk(parsed_directory):
#         for file in files:
#             if file.endswith('.json.gz'):
#                 file_path = os.path.join(root, file)
#                 file_paths.append(file_path)

#     # Use ThreadPoolExecutor to load files in parallel
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         future_to_file = {executor.submit(load_compressed_json, file_path): file_path for file_path in file_paths}

#         for future in tqdm.tqdm(as_completed(future_to_file), total=len(future_to_file), desc="Loading and batching files"):
#             file_path = future_to_file[future]
#             try:
#                 data = future.result()  # Load the file's data
#                 data_size = sys.getsizeof(json.dumps(data))  # Get the size of this data in bytes

#                 # If the current batch size plus new data exceeds the limit, process the batch
#                 if (batch_size + data_size) > max_batch_size_bytes:
#                     # Call your Milvus insertion function here
#                     index_batch_into_milvus(batch_data)  # Replace this with your Milvus indexing function

#                     # Reset the batch
#                     batch_data = []
#                     batch_size = 0

#                 # Add the data to the current batch
#                 batch_data.extend(data)  # Assuming data is a list of documents
#                 batch_size += data_size

#             except Exception as e:
#                 print(f"Error loading file {file_path}: {e}")

#     # Process any remaining data in the last batch
#     if batch_data:
#         index_batch_into_milvus(batch_data)  # Replace this with your Milvus indexing function

# def index_batch_into_milvus(batch_data):
#     print(f"Indexing batch with {len(batch_data)} documents into Milvus...")

#     entities = sbert_embeddings(batch_data)

#     # Verify the lengths of each component to ensure they match
#     print(f"Length of IDs: {len(entities[0])}")
#     print(f"Length of texts: {len(entities[1])}")
#     print(f"Shape of dense vectors: {len(entities[2])}")

#     col.insert(entities)
#     col.flush()

In [17]:
import os
import gzip
import json
import time
import tqdm
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed

# Function to load data from a compressed JSON file
def load_compressed_json(file_path):
    """Function to load data from a compressed JSON file."""
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        return json.load(f)

# Function to load data in 64MB batches for Milvus indexing
def load_data_in_batches_for_indexing(parsed_directory, max_batch_size_mb=64, max_workers=4):
    """Load data from compressed JSON files in batches for indexing into Milvus."""
    max_batch_size_bytes = max_batch_size_mb * 1024 * 1024  # Convert MB to bytes
    batch_data = []  # To store the current batch
    batch_size = 0  # To track the size of the current batch in bytes
    start_time = time.time()  # Record the start time

    # Collect all the .json.gz file paths
    file_paths = []
    for root, dirs, files in os.walk(parsed_directory):
        for file in files:
            if file.endswith('.json.gz'):
                file_path = os.path.join(root, file)
                file_paths.append(file_path)

    total_files = len(file_paths)
    checkpoint_times = {}  # Dictionary to save checkpoint times

    # Use ThreadPoolExecutor to load files in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_file = {executor.submit(load_compressed_json, file_path): file_path for file_path in file_paths}

        for idx, future in enumerate(tqdm.tqdm(as_completed(future_to_file), total=total_files, desc="Loading and batching files")):
            file_path = future_to_file[future]
            try:
                data = future.result()  # Load the file's data
                data_size = sys.getsizeof(json.dumps(data))  # Get the size of this data in bytes

                # If the current batch size plus new data exceeds the limit, process the batch
                if (batch_size + data_size) > max_batch_size_bytes:
                    # Call your Milvus insertion function here
                    index_batch_into_milvus(batch_data)  # Replace this with your Milvus indexing function

                    # Reset the batch
                    batch_data = []
                    batch_size = 0

                # Add the data to the current batch
                batch_data.extend(data)  # Assuming data is a list of documents
                batch_size += data_size

                # Record the time at each 10% completion
                percentage_complete = ((idx + 1) / total_files) * 100
                if percentage_complete >= 10 and (int(percentage_complete) % 10 == 0) and (int(percentage_complete) not in checkpoint_times):
                    elapsed_time = time.time() - start_time
                    checkpoint_times[int(percentage_complete)] = elapsed_time
                    print(f"Indexed {int(percentage_complete)}% of documents in {elapsed_time:.2f} seconds")

            except Exception as e:
                print(f"Error loading file {file_path}: {e}")

    # Process any remaining data in the last batch
    if batch_data:
        index_batch_into_milvus(batch_data)  # Replace this with your Milvus indexing function

    # Save checkpoint times to a file for future use
    with open('checkpoint_times.json', 'w') as f:
        json.dump(checkpoint_times, f)
    print("Checkpoint times saved to 'checkpoint_times.json'.")

def index_batch_into_milvus(batch_data):
    print(f"Indexing batch with {len(batch_data)} documents into Milvus...")

    entities = sbert_embeddings(batch_data)

    # Verify the lengths of each component to ensure they match
    print(f"Length of IDs: {len(entities[0])}")
    print(f"Length of texts: {len(entities[1])}")
    print(f"Shape of dense vectors: {len(entities[2])}")

    col.insert(entities)
    col.flush()

In [18]:
# run the function to load data in batches

load_data_in_batches_for_indexing('/home/sbasir/Thesis/Thesis/cp', max_batch_size_mb=64, max_workers=4)

Loading and batching files: 100%|██████████| 3/3 [00:00<00:00, 288.02it/s]

Indexed 100% of documents in 0.02 seconds
Indexing batch with 835 documents into Milvus...
{'id': '/5/URN_NBN_SI_IMG_APJFYR3F', 'text': "timestamp_update is 2019-07-08T19:33:17.185Z | type is IMAGE | content_tier is 0 | metadata_tier is A | edm:dataProvider is National and University Library, Ljubljana | National and University Library of Slovenia | edm:provider is Slovenski nacionalni agregator e-vsebin | Slovenian National E-content Aggregator | dc:description is Rokopis ni Kopitarjev, temveč je prepis Kopitarjeve razprave o jeziku današnjih štajerskih Slovencev po najstarejših staroslovanskih tekstih. Pisava po 24 vrstic&nbsp;na strani, žig ljubljanske licejske knjižnice pa je na ff. 1 in 32'.Cod. Kop. 27 | dc:format is 64 str. (32 f.) | dc:language is lat | dc:source is Kopitarjeva zbirka slovanskih kodeksov | dc:subject is rokopisi | dc:title is Kopitarjeva razprava o jeziku štajerskih Slovencev v razmerju do Clozovega zbornika | dc:type is rokopisi | dcterms:issued is 1836 | dc:s

/home/sbasir/Thesis/myenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Length of IDs: 835
Length of texts: 835
Shape of dense vectors: 835
Checkpoint times saved to 'checkpoint_times.json'.


In [10]:
query = "Vermeer"
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')
query_embeddings = model.encode(query)
k=10

# display(query_embeddings)

# print(query_embeddings)

search_params = {"metric_type": "IP"}

# Search topK docs based on dense and sparse vectors and rerank with RRF.
res = col.search(
    data = [query_embeddings],
    anns_field="dense_vector",
    param=search_params,
    limit=10,
    output_fields=["pk","text"]
)

for result in res[0]:
    print(result)

/home/sbasir/Thesis/myenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


id: /10/id_gvn_KONB14SMC_K0059, distance: 0.09831207990646362, entity: {'pk': '/10/id_gvn_KONB14SMC_K0059', 'text': "timestamp_update is 2019-06-19T08:56:05.851Z | type is IMAGE | content_tier is 4 | metadata_tier is A | edm:dataProvider is KB, National Library of the Netherlands | Koninklijke Bibliotheek | edm:provider is KB, National Library of the Netherlands | Koninklijke Bibliotheek | dc:description is Centsprent met 4 x 4 niet omkaderde houtsneden, met afbeeldingen van vogels: arend, kuifarend, struisvogel, fazant, ekster, papegaai, kruipeend, kraanvogel, pauw, steenuil, Indische mus, kalkoen, haan, gans, korhaan, zwaan. Bevat vierregelige rijmende onderschriften. | dc:subject is Ziet hier al weer een nieuwe prent, die m'om uw leerzaamheid nu schekt | Literatuur | Kunst en cultuur | Centsprenten | Vogels | dc:title is Ziet hier al weer een nieuwe prent, die m'om uw leerzaamheid nu schekt | dcterms:created is tussen 1819-1840 | dcterms:medium is 16 houtsn | edm:currentLocation is 